In [2]:
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path

cm_files = sorted(Path("output_files").glob("confusion_matrices_*.csv"))

datasets = {}
for f in cm_files:
    label = f.stem.replace("confusion_matrices_", "")
    df = pd.read_csv(f)
    cm_cols = [c for c in df.columns if c.startswith("true_") and "_pred_" in c]
    df[cm_cols] = df[cm_cols].div(df[cm_cols].sum(axis=1), axis=0)
    datasets[label] = (df, cm_cols)

fig = go.Figure()
labels = list(datasets.keys())
trace_counts = []

for label, (df, cm_cols) in datasets.items():
    for col in cm_cols:
        fig.add_trace(go.Scatter(
            x=df["chunk_idx"],
            y=df[col],
            mode="lines+markers",
            name=col,
            visible=(label == labels[0]),
        ))
    trace_counts.append(len(cm_cols))

buttons = []
for i, label in enumerate(labels):
    visibility = []
    for j, count in enumerate(trace_counts):
        visibility.extend([i == j] * count)
    buttons.append(dict(label=label, method="update", args=[
        {"visible": visibility},
        {"title": f"Confusion Matrix Values Over Time — {label}"},
    ]))

fig.update_layout(
    title=f"Confusion Matrix Values Over Time — {labels[0]}",
    xaxis_title="Chunk Index",
    yaxis_title="Proportion",
    legend_title="CM Cell",
    hovermode="x unified",
    template="plotly_white",
    updatemenus=[dict(
        active=0,
        buttons=buttons,
        x=0.0,
        xanchor="left",
        y=1.15,
        yanchor="top",
    )],
)
fig.show()